In [55]:
!pip install -q pdfplumber

In [56]:
import pdfplumber

# Load PDF data

In [57]:
def load_pdf(file_path):
    all_text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                all_text += text + "\n"
    return all_text

In [58]:
path = r'/content/company_policies.pdf'
text = load_pdf(path)
text

'Company Policies\nLeave Policy\n• Employees receive 12 casual leaves annually.\n• Employees receive 15 sick leaves annually.\n• Unused casual leaves cannot be carried forward.\nWork From Home Policy\n• Employees may work from home twice per week.\n• Manager approval is required for additional remote work.\nTravel Policy\n• Travel expenses are reimbursed within 30 days.\n• Original receipts must be submitted for reimbursement.\nMedical Insurance Policy\n• All employees are covered under company medical insurance.\n• Dependent coverage is available for spouse and children.\n'

In [59]:
# Info
print('Document text:\n\n', text)
print('# characters:', len(text))
print('# words:', len(text.split()))

Document text:

 Company Policies
Leave Policy
• Employees receive 12 casual leaves annually.
• Employees receive 15 sick leaves annually.
• Unused casual leaves cannot be carried forward.
Work From Home Policy
• Employees may work from home twice per week.
• Manager approval is required for additional remote work.
Travel Policy
• Travel expenses are reimbursed within 30 days.
• Original receipts must be submitted for reimbursement.
Medical Insurance Policy
• All employees are covered under company medical insurance.
• Dependent coverage is available for spouse and children.

# characters: 565
# words: 87


### Clean the Extracted Text

Now, let's clean the extracted text by:
- Removing leading/trailing whitespace from each line.
- Removing empty lines.
- Joining the lines back into a single string.
- Converting the text to lowercase.

In [60]:
# Split the text into lines and clean each line
cleaned_lines = []
for line in text.split('\n'):
    # First remove the bullet point, then strip whitespace
    temp_line = line.replace('•', '').strip()
    if temp_line:
        cleaned_lines.append(temp_line.lower()) # Convert to lowercase

# Join the cleaned lines back into a single string
cleaned_text = ' '.join(cleaned_lines)

print('Cleaned Document Text (first 500 characters):\n', cleaned_text[:500], '...')
print('\n# characters after cleaning:', len(cleaned_text))
print('# words after cleaning:', len(cleaned_text.split()))

Cleaned Document Text (first 500 characters):
 company policies leave policy employees receive 12 casual leaves annually. employees receive 15 sick leaves annually. unused casual leaves cannot be carried forward. work from home policy employees may work from home twice per week. manager approval is required for additional remote work. travel policy travel expenses are reimbursed within 30 days. original receipts must be submitted for reimbursement. medical insurance policy all employees are covered under company medical insurance. dependent  ...

# characters after cleaning: 546
# words after cleaning: 78


# Chunking

## Fixed size chunking

In [61]:
chunk_size = [100, 200, 300]
all_chunks = []

for size in chunk_size:
  chunks = [cleaned_text[i:i+size] for i in range(0, len(text), size)]
  all_chunks.append(chunks)

# print chunks
for i in range(len(chunk_size)):
  print('CHUNK SIZE:', chunk_size[i])
  for chunk in all_chunks[i]:
    print(chunk)
    print('-'*50)
  print('#'*50)
  print()

CHUNK SIZE: 100
company policies leave policy employees receive 12 casual leaves annually. employees receive 15 sick
--------------------------------------------------
 leaves annually. unused casual leaves cannot be carried forward. work from home policy employees ma
--------------------------------------------------
y work from home twice per week. manager approval is required for additional remote work. travel pol
--------------------------------------------------
icy travel expenses are reimbursed within 30 days. original receipts must be submitted for reimburse
--------------------------------------------------
ment. medical insurance policy all employees are covered under company medical insurance. dependent 
--------------------------------------------------
coverage is available for spouse and children.
--------------------------------------------------
##################################################

CHUNK SIZE: 200
company policies leave policy employees receive 12 casual 

## Recursive chunking

In [62]:
!pip install -q langchain langchain-text-splitters

In [63]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size[0],
    chunk_overlap=10,
)

chunks = splitter.split_text(cleaned_text)

print('CHUNK SIZE:', chunk_size[0])
for chunk in chunks:
  print(chunk)
  print('-'*50)

CHUNK SIZE: 100
company policies leave policy employees receive 12 casual leaves annually. employees receive 15 sick
--------------------------------------------------
15 sick leaves annually. unused casual leaves cannot be carried forward. work from home policy
--------------------------------------------------
policy employees may work from home twice per week. manager approval is required for additional
--------------------------------------------------
remote work. travel policy travel expenses are reimbursed within 30 days. original receipts must be
--------------------------------------------------
must be submitted for reimbursement. medical insurance policy all employees are covered under
--------------------------------------------------
under company medical insurance. dependent coverage is available for spouse and children.
--------------------------------------------------


## Sentence based chunking

In [64]:
!pip install nltk -q
import nltk
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_r

True

In [65]:
from nltk.tokenize import sent_tokenize

chunks = sent_tokenize(cleaned_text)

for chunk in chunks:
  print(chunk)
  print('-'*50)

company policies leave policy employees receive 12 casual leaves annually.
--------------------------------------------------
employees receive 15 sick leaves annually.
--------------------------------------------------
unused casual leaves cannot be carried forward.
--------------------------------------------------
work from home policy employees may work from home twice per week.
--------------------------------------------------
manager approval is required for additional remote work.
--------------------------------------------------
travel policy travel expenses are reimbursed within 30 days.
--------------------------------------------------
original receipts must be submitted for reimbursement.
--------------------------------------------------
medical insurance policy all employees are covered under company medical insurance.
--------------------------------------------------
dependent coverage is available for spouse and children.
---------------------------------------------

## Semantic chunking

In [66]:
!pip install -q sentence-transformers

In [70]:
from sentence_transformers import SentenceTransformer

ImportError: cannot import name '_Ink' from 'PIL._typing' (/usr/local/lib/python3.12/dist-packages/PIL/_typing.py)

In [68]:

from sklearn.metrics.pairwise import cosine_similarity

sentences = sent_tokenize(text)
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(sentences)

threshold = 0.5

chunks = []
current_chunk = [sentences[0]]

for i in range(1, len(sentences)):
  similarity = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]
  if similarity < threshold:
    chunks.append(' '.join(current_chunk))
    current_chunk = [sentences[i]]
  else:
    chunks.append(' '.join(current_chunk))
    current_chunk = [sentences[i]]

chunks.append(' '.join(current_chunk))

for chunk in chunks:
  print(chunk)
  print('-'*50)

ImportError: cannot import name '_Ink' from 'PIL._typing' (/usr/local/lib/python3.12/dist-packages/PIL/_typing.py)